In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Callable
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [3]:
""""
Import two different tables: PLA_filter + configurations_matrix

PLA_filter: Phase 0, manufacturing restrictions
configurations_matrix: Contains the values of each mechanical system against criteria
"""

# PLA filter

PLA_filter = pd.read_csv("PLA_filter.csv",
    sep=";",
    index_col=0
).dropna(how="all") #remove rows or columns where absolutely all values are null (NaN or None)

# Matrix Mechanical systems vs Criteria

matrix = pd.read_csv(
    "configurations_matrix.csv",
    sep=";",
    index_col=0
).dropna(how="all")

print("PLA FILTER TABLE")
display(PLA_filter)

print("\nCONFIGURATIONS MATRIX")
display(matrix)


PLA FILTER TABLE


,Cable-pulley / tendon-driven transmission,Bowden cable / cable-conduit transmission,Elastic cable / spring-assisted transmission,Articulated hinge with passive element,Flexible shell / leaf-spring / strut mechanism,Hybrid rigid-compliant mechanism,Spring-loaded linkage / four-bar / parallelogram,Cam-spring-cable / variable moment arm mechanism,Local rigid rotary transmission,Linear-to-rotary transmission
PLA_filter,,,,,,,,,,
PLA_fully_printable,0,0,0,0,1,0,0,0,0,0
PLA_Needs_simple_components,1,1,1,1,1,0,1,1,1,0
PLA_Needs_complex_components,1,1,0,0,0,1,0,1,1,1



CONFIGURATIONS MATRIX


,Cable-pulley / tendon-driven transmission,Bowden cable / cable-conduit transmission,Elastic cable / spring-assisted transmission,Articulated hinge with passive element,Flexible shell / leaf-spring / strut mechanism,Hybrid rigid-compliant mechanism,Spring-loaded linkage / four-bar / parallelogram,Cam-spring-cable / variable moment arm mechanism,Local rigid rotary transmission,Linear-to-rotary transmission
Criteria,,,,,,,,,,
Assistance_constant,2,3,1,2,1,2,2,1,3,2
Assistance_variable_with_angle,2,2,2,2,2,2,2,3,1,2
Assistance_partial_range,2,2,2,3,3,1,2,3,1,1
Remote_transmission,3,3,2,1,1,1,1,3,1,1
Low_distal_mass_suitability,3,3,3,1,3,2,1,3,1,1
Adaptability_to_misalignments,2,2,3,1,2,3,2,2,1,1
Motion_precision,2,2,2,3,2,1,3,2,3,3
Supports_rotation,2,2,2,3,2,2,3,2,3,1
Supports_translation,2,3,2,1,2,2,2,2,1,3


In [4]:
"""
Define base weights (1.0) for each criteria. At the beggining every criteria has
the same importance. Rules will modify these weights according to the case
"""

base_weights = pd.Series(1.0, index=matrix.index).to_dict()

weights = pd.DataFrame({
    'Criteria': list(base_weights.keys()),
    'Weight': list(base_weights.values())
})
print("BASE WEIGHTS TABLE")
display(weights)

BASE WEIGHTS TABLE


,Criteria,Weight
0,Assistance_constant,1.0
1,Assistance_variable_with_angle,1.0
2,Assistance_partial_range,1.0
3,Remote_transmission,1.0
4,Low_distal_mass_suitability,1.0
5,Adaptability_to_misalignments,1.0
6,Motion_precision,1.0
7,Supports_rotation,1.0
8,Supports_translation,1.0
9,Supports_combined_motion,1.0


In [5]:
# PHASE 0 - MANUFACTURING FILTERS

def apply_manufacturing_filter(
    pla_filter: pd.DataFrame,
    integration_level: str
) -> tuple[list[str], list[str], list[str]]:
    """
    Apply manufacturing feasibility filters.

    Returns
    -------
    tuple
        - eligible_configs
        - eliminated_configs
        - complexity_adjusted_configs
    """

    eligible_configs = list(pla_filter.columns)
    eliminated_configs = []
    complexity_adjusted_configs = []

    if integration_level == "fully_printable":
        row = pla_filter.loc["PLA_fully_printable"]
        eligible_configs = [c for c in pla_filter.columns if row[c] == 1]
        eliminated_configs = [c for c in pla_filter.columns if row[c] != 1]

    elif integration_level == "simple_components":
        simple_row = pla_filter.loc["PLA_Needs_simple_components"]
        complex_row = pla_filter.loc["PLA_Needs_complex_components"]

        eligible_configs = [c for c in pla_filter.columns if simple_row[c] == 1]
        eliminated_configs = [c for c in pla_filter.columns if simple_row[c] != 1]

        # These remain eligible, but their complexity score will be adjusted
        complexity_adjusted_configs = [
            c for c in eligible_configs if complex_row[c] == 1
        ]

    elif integration_level == "complex_components":
        pass

    return eligible_configs, eliminated_configs, complexity_adjusted_configs

def apply_manufacturing_complexity_penalty(
    scores: dict[str, float],
    score_breakdown: dict[str, dict],
    complexity_adjusted_configs: list[str],
    penalty_factor: float = 0.80
) -> dict[str, float]:
    """
    Apply rule 0-R2: penalize the Complexity_level contribution (not the
    matrix value) for configurations that require complex components under
    a manufacturing level that only allows simple ones.

    This preserves the original matrix values and applies the penalty at
    the score level, consistent with the methodology (weight-based penalty,
    not value mutation).

    Parameters
    ----------
    scores : dict
        Total score per configuration.
    score_breakdown : dict
        Per-criterion contributions per configuration.
    complexity_adjusted_configs : list
        Configurations that should receive the complexity penalty.
    penalty_factor : float
        Multiplier applied to the Complexity_level contribution (default 0.80).

    Returns
    -------
    dict
        Updated scores with penalty applied.
    """
    adjusted_scores = scores.copy()

    for config in complexity_adjusted_configs:
        if config in score_breakdown and "Complexity_level" in score_breakdown[config]:
            original_contribution = score_breakdown[config]["Complexity_level"]
            penalty = original_contribution * (1.0 - penalty_factor)
            adjusted_scores[config] -= penalty

    return adjusted_scores

In [6]:
def apply_architecture_branching(
    filtered_matrix: pd.DataFrame,
    user_inputs: dict
) -> tuple[pd.DataFrame, list[str]]:
    """
    Apply architectural branching before score calculation.

    Logic:
    - If remote transmission IS required: do not exclude local candidates.
      Let the weighting of Remote_transmission decide.
    - If remote transmission is NOT required: exclude clearly remote families.
    - If active actuation is required: keep only active-compatible candidates.
    - If passive actuation is selected: do not exclude candidates here;
      let the weights and matrix values decide.
    """

    candidate_configs = list(filtered_matrix.columns)
    excluded_configs = []

    # ----------------------------------------------------------
    # 1. Remote vs local branching
    # ----------------------------------------------------------
    if user_inputs.get("C2_remote_transmission") == "no":
        # New 1-3 scale: 1 = local, 2 = partially remote, 3 = remote
        # Exclude clearly remote mechanisms (value 3); keep local (1) and hybrid (2)
        keep = [
            config for config in candidate_configs
            if filtered_matrix.loc["Remote_transmission", config] <= 2
        ]

        if len(keep) > 0:
            excluded_configs.extend(
                [config for config in candidate_configs if config not in keep]
            )
            candidate_configs = keep

    # If C2_remote_transmission == "yes":
    # do NOT filter; let the criterion weights handle it

    # ----------------------------------------------------------
    # 2. Passive vs active branching
    # ----------------------------------------------------------
    if user_inputs.get("Actuation_type") == "active":
        if "Active_actuation_compatibility" in filtered_matrix.index:
            # New 1-3 scale: 1 = incompatible, 2 = compatible, 3 = optimal
            # Exclude incompatible mechanisms (value 1); keep compatible (2) and optimal (3)
            keep = [
                config for config in candidate_configs
                if filtered_matrix.loc["Active_actuation_compatibility", config] >= 2
            ]

            if len(keep) > 0:
                excluded_configs.extend(
                    [config for config in candidate_configs if config not in keep]
                )
                candidate_configs = keep

    # If passive: no exclusion here

    branched_matrix = filtered_matrix[candidate_configs]

    return branched_matrix, excluded_configs

In [7]:
# =============================================================================
# PHASE 1 - RULE DEFINITION
# =============================================================================

@dataclass
class Rule:
    """
    Structure representing one rule for weight adjustment.
    """
    id: str
    description: str
    condition: Callable[[dict], bool]
    adjustments: dict[str, float]   # {criterion: multiplier}


def define_all_rules() -> list[Rule]:
    """
    Define all DSS rules according to the methodology.

    Returns
    -------
    list[Rule]
        List of rule objects.
    """

    rules = []

    # ==================== SECTION A: SOFT RULES ====================

    # A1: What movement do you want to assist?
  
    rules.append(Rule(
        id="A1-R1",
        description="Combination -> Supports_combined_motion x1.20",
        condition=lambda u: u.get("A1_movement") == "combination",
        adjustments={"Supports_combined_motion": 1.20}
    ))

    rules.append(Rule(
        id="A1-R2",
        description="Elevation/complex motion -> Combined x1.20, Adaptability x1.15",
        condition=lambda u: u.get("A1_movement") in ["elevation", "complex_spatial"],
        adjustments={
            "Supports_combined_motion": 1.20,
            "Adaptability_to_misalignments": 1.15
        }
    ))

    # A2: Required ROM
    rules.append(Rule(
        id="A2-R1",
        description="Reduced ROM -> Motion_precision x1.10",
        condition=lambda u: u.get("A2_rom") == "reduced",
        adjustments={"Motion_precision": 1.10}
    ))

    rules.append(Rule(
        id="A2-R2",
        description="Wide ROM -> Adaptability x1.20",
        condition=lambda u: u.get("A2_rom") == "wide",
        adjustments={
            "Adaptability_to_misalignments": 1.20
        }
    ))

   
    # A3: Uniaxial or multiplanar
    rules.append(Rule(
        id="A3-R1",
        description="Uniaxial -> Supports_rotation x1.20",
        condition=lambda u: u.get("A3_motion_type") == "uniaxial",
        adjustments={"Supports_rotation": 1.20}
    ))

    rules.append(Rule(
        id="A3-R2",
        description="Multiplanar -> Combined x1.20, Adaptability x1.20",
        condition=lambda u: u.get("A3_motion_type") == "multiplanar",
        adjustments={
            "Supports_combined_motion": 1.20,
            "Adaptability_to_misalignments": 1.20
        }
    ))

    # ==================== SECTION B: ORTHOSIS FUNCTION ====================
    # B1: Action to realize
    rules.append(Rule(
        id="B1-R1",
        description="Assist -> all assistance types x1.20",
        condition=lambda u: u.get("B1_action") == "assist",
        adjustments={
            "Assistance_constant": 1.10,
            "Assistance_variable_with_angle": 1.10,
            "Assistance_partial_range": 1.10
        }
    ))

    rules.append(Rule(
        id="B1-R2a",
        description="Assist + force + low distal mass -> Low_distal_mass_suitability x1.20",
        condition=lambda u: (
            u.get("B1_action") == "assist" and
            u.get("B2_force_required") == "yes" and
            u.get("C3_low_distal_mass") == "yes"
        ),
        adjustments={
            "Low_distal_mass_suitability": 1.20
        }
    ))

    rules.append(Rule(
        id="B1-R2b",
        description="Assist + force + remote required -> Remote_transmission x1.20",
        condition=lambda u: (
            u.get("B1_action") == "assist" and
            u.get("B2_force_required") == "yes" and
            u.get("C2_remote_transmission") == "yes"
        ),
        adjustments={
            "Remote_transmission": 1.20
        }
    ))
    
    
    rules.append(Rule(
        id="B1-R3",
        description="Resist -> Precision x1.50, motion support x1.20, Remote x0.80",
        condition=lambda u: u.get("B1_action") == "resist",
        adjustments={
            "Motion_precision": 1.50,
            "Supports_rotation": 1.20,
            "Supports_translation": 1.20,
            "Supports_combined_motion": 1.20,
            "Remote_transmission": 0.80
        }
    ))

    rules.append(Rule(
        id="B1-R4",
        description="Guide -> Precision x2.00, Supports_rot/trans/comb x1.50, Adaptability x1.20",
        condition=lambda u: u.get("B1_action") == "guide",
        adjustments={
            "Motion_precision": 2.00,
            "Supports_rotation": 1.50,
            "Supports_translation": 1.50,
            "Supports_combined_motion": 1.50,
            "Adaptability_to_misalignments": 1.20
        }
    ))

    rules.append(Rule(
        id="B1-R5",
        description="Return to position -> Partial_range x1.30, Constant x1.10, Low_mass x1.15",
        condition=lambda u: u.get("B1_action") == "return_to_position",
        adjustments={
            "Assistance_partial_range": 1.30,
            "Assistance_constant": 1.10,
            "Low_distal_mass_suitability": 1.15
        }
    ))
    rules.append(Rule(
        id="B1-R6",
        description="Maintain trajectory -> Motion_precision x2.00, Adaptability x1.20",
        condition=lambda u: u.get("B1_action") == "maintain_trajectory",
        adjustments={
            "Motion_precision": 2.00,
            "Adaptability_to_misalignments": 1.20
        }
    ))

    # B2: Force/torque generation
   

    rules.append(Rule(
        id="B2-R1",
        description="No force -> Precision x1.20", 
        condition=lambda u: u.get("B2_force_required") == "no",
        adjustments={
            "Motion_precision": 1.20
        }
    ))

    # ==================== SECTION C: TECHNICAL REQUIREMENTS ====================

    # C1: Assistance type
    rules.append(Rule(
        id="C1-R1",
        description="Constant assistance -> Constant x2.00, others x0.80",
        condition=lambda u: u.get("C1_assistance_type") == "constant",
        adjustments={
            "Assistance_constant": 2.00,
            "Assistance_variable_with_angle": 0.50,
            "Assistance_partial_range": 0.50
        }
    ))

    rules.append(Rule(
        id="C1-R2",
        description="Variable with angle -> Variable x2.00",
        condition=lambda u: u.get("C1_assistance_type") == "variable_with_angle",
        adjustments={
            "Assistance_variable_with_angle": 2.00,
            "Assistance_constant": 0.50,
            "Assistance_partial_range": 0.50
        }
    ))

    rules.append(Rule(
        id="C1-R3",
        description="Partial range -> Partial x2.00",
        condition=lambda u: u.get("C1_assistance_type") == "partial_range",
        adjustments={
            "Assistance_partial_range": 2.00,
            "Assistance_constant": 0.50,
            "Assistance_variable_with_angle": 0.50
            
        }
    ))

    # C2: Remote transmission
    rules.append(Rule(
        id="C2-R1",
        description="Remote required -> Remote x2.00, Low_mass x1.50, Adaptability x1.20, Motion_precision x0.95",
        condition=lambda u: u.get("C2_remote_transmission") == "yes",
        adjustments={
            "Remote_transmission": 2.00,
            "Low_distal_mass_suitability": 1.50,
            "Adaptability_to_misalignments": 1.20,
            "Motion_precision": 0.95
        }
    ))
    

    rules.append(Rule(
        id="C2-R2",
        description="No remote -> Remote x0.50, Precision x1.20",
        condition=lambda u: u.get("C2_remote_transmission") == "no",
        adjustments={
            "Remote_transmission": 0.50,
            "Motion_precision": 1.20
        }
    ))


    # C3: Low distal mass
    rules.append(Rule(
        id="C3-R1",
        description="Low mass critical -> Low_mass x2.00, Remote x1.30, Complexity x0.90",
        condition=lambda u: u.get("C3_low_distal_mass") == "yes",
        adjustments={
            "Low_distal_mass_suitability": 2.00,
            "Remote_transmission": 1.30,
            "Complexity_level": 0.90
        }
    ))

   

    # C4: Adaptability
    rules.append(Rule(
        id="C4-R1",
        description="High adaptability -> Adaptability x2.00, Combined x1.20, Precision x0.90",
        condition=lambda u: u.get("C4_adaptability") == "high",
        adjustments={
            "Motion_precision": 0.90,
            "Supports_combined_motion": 1.20,
            "Adaptability_to_misalignments": 2.00
        }
    ))

    rules.append(Rule(
        id="C4-R2",
        description="Medium adaptability -> Adaptability x1.50",
        condition=lambda u: u.get("C4_adaptability") == "medium",
        adjustments={"Adaptability_to_misalignments": 1.50}
    ))

    rules.append(Rule(
        id="C4-R3",
        description="Low adaptability -> Adaptability x0.80, Precision x1.30",
        condition=lambda u: u.get("C4_adaptability") == "low",
        adjustments={
            "Adaptability_to_misalignments": 0.80,
            "Motion_precision": 1.30
        }
    ))

    # C5: Motion precision
    rules.append(Rule(
        id="C5-R1",
        description="High precision -> Precision x2.00, Remote x0.90, Adaptability x0.90",
        condition=lambda u: u.get("C5_precision") == "high",
        adjustments={
            "Motion_precision": 2.00,
            "Remote_transmission": 0.90,
            "Adaptability_to_misalignments": 0.90
        }
    ))

  

    rules.append(Rule(
        id="C5-R2",
        description="Low precision -> Adaptability x1.20, Precision x0.80",
        condition=lambda u: u.get("C5_precision") == "low",
        adjustments={
            "Adaptability_to_misalignments": 1.20,
            "Motion_precision": 0.80
        }
    ))

    # C6: Mechanical movement type
    rules.append(Rule(
        id="C6-R1",
        description="Rotation -> Supports_rotation x2.00",
        condition=lambda u: u.get("C6_movement_type") == "rotation",
        adjustments={"Supports_rotation": 2.00}
    ))

    rules.append(Rule(
        id="C6-R2",
        description="Translation -> Supports_translation x2.00",
        condition=lambda u: u.get("C6_movement_type") == "translation",
        adjustments={"Supports_translation": 2.00}
    ))

    rules.append(Rule(
        id="C6-R3",
        description="Combined -> Supports_combined_motion x2.00, Adaptability x1.20",
        condition=lambda u: u.get("C6_movement_type") == "combined",
        adjustments={
            "Supports_combined_motion": 2.00,
            "Adaptability_to_misalignments": 1.20
        }
    ))

    # C7: Acceptable complexity
    rules.append(Rule(
        id="C7-R1",
        description="Low complexity preferred -> Complexity x2.00",
        condition=lambda u: u.get("C7_complexity") == "low",
        adjustments={"Complexity_level": 2.00}
    ))

    rules.append(Rule(
        id="C7-R2",
        description="Medium complexity -> Complexity x1.20",
        condition=lambda u: u.get("C7_complexity") == "medium",
        adjustments={"Complexity_level": 1.20}
    ))
    
    rules.append(Rule(
        id="C7-R3",
        description="High complexity acceptable -> Complexity_level x0.70",
        condition=lambda u: u.get("C7_complexity") == "high",
        adjustments={"Complexity_level": 0.70}
    ))
    
    # C8: Passive or Active 
    
    rules.append(Rule(
        id="C8-R1",
        description="Active actuation required -> Active compatibility x2.00, Precision x1.20",
        condition=lambda u: u.get("Actuation_type") == "active",
        adjustments={"Active_actuation_compatibility": 2.00, "Motion_precision": 1.20}
    ))

    rules.append(Rule(
        id="C8-R2",
        description="Passive actuation -> Active compatibility x0.50",
        condition=lambda u: u.get("Actuation_type") == "passive",
        adjustments={
            "Active_actuation_compatibility": 0.50
        }
    ))
    # ==================== INTERACTION RULES ====================



    rules.append(Rule(
        id="I-R1a",
        description="Guide/Maintain + rotation -> Supports_rotation x1.10",
        condition=lambda u: (
            u.get("B1_action") in ["guide", "maintain_trajectory"] and
            u.get("C6_movement_type") == "rotation"
        ),
        adjustments={
            "Supports_rotation": 1.10
        }
    ))

    rules.append(Rule(
        id="I-R1b",
        description="Guide/Maintain + translation -> Supports_translation x1.10",
        condition=lambda u: (
            u.get("B1_action") in ["guide", "maintain_trajectory"] and
            u.get("C6_movement_type") == "translation"
        ),
        adjustments={
            "Supports_translation": 1.10
        }
    ))

    rules.append(Rule(
        id="I-R1c",
        description="Guide/Maintain + combined motion -> Supports_combined_motion x1.10",
        condition=lambda u: (
            u.get("B1_action") in ["guide", "maintain_trajectory"] and
            u.get("C6_movement_type") == "combined"
        ),
        adjustments={
            "Supports_combined_motion": 1.10
        }
    ))
    
   

    rules.append(Rule(
        id="I-R2a",
        description="Return + no remote + passive + low complexity -> Partial_range x1.20, Complexity x1.15",
        condition=lambda u: (
            u.get("B1_action") == "return_to_position" and
            u.get("C2_remote_transmission") == "no" and
            u.get("Actuation_type") == "passive" and
            u.get("C7_complexity") == "low"
        ),
        adjustments={
            "Assistance_partial_range": 1.20,
            "Complexity_level": 1.15
        }
    ))

    rules.append(Rule(
        id="I-R2b",
        description="Return + no remote + rotation -> Supports_rotation x1.15, Motion_precision x1.10",
        condition=lambda u: (
            u.get("B1_action") == "return_to_position" and
            u.get("C2_remote_transmission") == "no" and
            u.get("C6_movement_type") == "rotation"
        ),
        adjustments={
            "Supports_rotation": 1.15,
            "Motion_precision": 1.10
        }
    ))
    
   
    rules.append(Rule(
        id="I-R3",
        description="Uniaxial + no remote + rotation + low complexity -> Supports_rotation x1.20, Motion_precision x1.20",
        condition=lambda u: (
            u.get("A3_motion_type") == "uniaxial" and
            u.get("C2_remote_transmission") == "no" and
            u.get("C6_movement_type") == "rotation" and
            u.get("C7_complexity") == "low"
        ),
        adjustments={
            "Supports_rotation": 1.20,
            "Motion_precision": 1.20
        }
    ))
    
  
    rules.append(Rule(
        id="I-R4",
        description="Active + no remote + rotation -> Supports_rotation x1.15, Motion_precision x1.10",
        condition=lambda u: (
            u.get("Actuation_type") == "active" and
            u.get("C2_remote_transmission") == "no" and
            u.get("C6_movement_type") == "rotation"
        ),
        adjustments={
            "Supports_rotation": 1.15,
            "Motion_precision": 1.10
        }
    ))

    # I-R5: variable assist + remote + low mass → amplify differentiating criterion of A8
    rules.append(Rule(
        id="I-R5",
        description="Variable assist + remote + low mass -> Assistance_variable_with_angle x1.50",
        condition=lambda u: (
            u.get("C1_assistance_type") == "variable_with_angle" and
            u.get("C2_remote_transmission") == "yes" and
            u.get("C3_low_distal_mass") == "yes"
        ),
        adjustments={"Assistance_variable_with_angle": 1.50}
    ))

    # I-R5b: active + local + rotation → amplify differentiating criterion of A9
    rules.append(Rule(
        id="I-R5b",
        description="Active + local + rotation -> Active_actuation_compatibility x1.30",
        condition=lambda u: (
            u.get("Actuation_type") == "active" and
            u.get("C2_remote_transmission") == "no" and
            u.get("C6_movement_type") == "rotation"
        ),
        adjustments={"Active_actuation_compatibility": 1.30}
    ))

    return rules


# Create rule list
ALL_RULES = define_all_rules()

print(f"Defined rules: {len(ALL_RULES)}")

rules_summary = pd.DataFrame([
    {"ID": r.id, "Description": r.description}
    for r in ALL_RULES
])

display(rules_summary)


Defined rules: 42


,ID,Description
0,A1-R1,Combination -> Supports_combined_motion x1.20
1,A1-R2,"Elevation/complex motion -> Combined x1.20, Ad..."
2,A2-R1,Reduced ROM -> Motion_precision x1.10
3,A2-R2,Wide ROM -> Adaptability x1.20
4,A3-R1,Uniaxial -> Supports_rotation x1.20
5,A3-R2,"Multiplanar -> Combined x1.20, Adaptability x1.20"
6,B1-R1,Assist -> all assistance types x1.20
7,B1-R2a,Assist + force + low distal mass -> Low_distal...
8,B1-R2b,Assist + force + remote required -> Remote_tra...
9,B1-R3,"Resist -> Precision x1.50, motion support x1.2..."


In [8]:

# =============================================================================
# RULE ENGINE - APPLY RULES AND CALCULATE FINAL WEIGHTS
# =============================================================================

class RuleEngine:
    """
    Rule engine that applies active rule multipliers and calculates
    final normalized criterion weights.
    """

    UPPER_CAP = 3.00   # upper limit = 300% of base weight
    LOWER_CAP = 0.50   # lower limit = 50% of base weight

    def __init__(self, base_weights: dict[str, float], rules: list[Rule]):
        self.base_weights = base_weights.copy()
        self.rules = rules

    def apply_rules(self, user_inputs: dict) -> tuple:
        """
        Apply all active rules and calculate final weights.

        Parameters
        ----------
        user_inputs : dict
            Dictionary containing the answers of the user/case.

        Returns
        -------
        tuple
            A tuple with:
            - normalized_weights: final normalized weights
            - activated_rules: list of activated Rule objects
            - detail: DataFrame with full weight calculation trace
        """

        # 1. Initialize accumulated multipliers
        accumulated_multipliers = {
            criterion: 1.0 for criterion in self.base_weights
        }

        activated_rules = []

        # 2. Evaluate all rules
        for rule in self.rules:
            try:
                if rule.condition(user_inputs):
                    activated_rules.append(rule)

                    for criterion, multiplier in rule.adjustments.items():
                        if criterion in accumulated_multipliers:
                            accumulated_multipliers[criterion] *= multiplier

            except Exception as e:
                print(f"Error in rule {rule.id}: {e}")

        # 3. Raw weights
        raw_weights = {
            criterion: self.base_weights[criterion] * accumulated_multipliers[criterion]
            for criterion in self.base_weights
        }

        # 4. Apply lower/upper caps
        capped_weights = {}

        for criterion, raw_value in raw_weights.items():
            base_value = self.base_weights[criterion]
            lower_limit = self.LOWER_CAP * base_value
            upper_limit = self.UPPER_CAP * base_value

            capped_weights[criterion] = max(lower_limit, min(raw_value, upper_limit))

        # 5. Normalize weights
        total_weight = sum(capped_weights.values())

        normalized_weights = {
            criterion: value / total_weight
            for criterion, value in capped_weights.items()
        }

        # 6. Create trace table for transparency
        detail = pd.DataFrame({
            "Criterion": list(self.base_weights.keys()),
            "Base_weight": [self.base_weights[c] for c in self.base_weights],
            "Accumulated_multiplier": [accumulated_multipliers[c] for c in self.base_weights],
            "Raw_weight": [raw_weights[c] for c in self.base_weights],
            "Capped_weight": [capped_weights[c] for c in self.base_weights],
            "Normalized_weight": [normalized_weights[c] for c in self.base_weights]
        })

        return normalized_weights, activated_rules, detail


# Create rule engine instance
rule_engine = RuleEngine(base_weights, ALL_RULES)
print("Rule engine initialized correctly")





normalized_weights, activated_rules, detail = rule_engine.apply_rules({
    "A1_movement": "flexion_extension",
    "A2_rom": "medium",
    "A3_motion_type": "uniaxial",
    "B1_action": "assist",
    "B2_force_required": "yes",
    "C1_assistance_type": "constant",
    "C2_remote_transmission": "no",
    "C3_low_distal_mass": "yes",
    "C4_adaptability": "medium",
    "C5_precision": "medium",
    "C6_movement_type": "rotation",
    "C7_complexity": "low"
})

print("Activated rules:", [r.id for r in activated_rules])
display(detail)

Rule engine initialized correctly
Activated rules: ['A3-R1', 'B1-R1', 'B1-R2a', 'C1-R1', 'C2-R2', 'C3-R1', 'C4-R2', 'C6-R1', 'C7-R1', 'I-R3']


,Criterion,Base_weight,Accumulated_multiplier,Raw_weight,Capped_weight,Normalized_weight
0,Assistance_constant,1.0,2.20,2.20,2.20,0.129641
1,Assistance_variable_with_angle,1.0,0.55,0.55,0.55,0.032410
2,Assistance_partial_range,1.0,0.55,0.55,0.55,0.032410
3,Remote_transmission,1.0,0.65,0.65,0.65,0.038303
4,Low_distal_mass_suitability,1.0,2.40,2.40,2.40,0.141426
5,Adaptability_to_misalignments,1.0,1.50,1.50,1.50,0.088391
6,Motion_precision,1.0,1.44,1.44,1.44,0.084856
7,Supports_rotation,1.0,2.88,2.88,2.88,0.169711
8,Supports_translation,1.0,1.00,1.00,1.00,0.058928
9,Supports_combined_motion,1.0,1.00,1.00,1.00,0.058928


In [9]:

# =============================================================================
# PHASE 2 - SCORE CALCULATION AND RANKING
# =============================================================================

def calculate_scores(
    filtered_matrix: pd.DataFrame,
    normalized_weights: dict[str, float],
    full_matrix: pd.DataFrame | None = None
) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    """
    Calculate final scores and criterion contributions.

    Parameters
    ----------
    filtered_matrix : pd.DataFrame
        Matrix containing only the eligible configurations after Phase 0
        and architecture branching.
    normalized_weights : dict
        Normalised criterion weights from the rule engine.
    full_matrix : pd.DataFrame, optional
        Complete unfiltered matrix used to compute the global maximum per
        criterion for min-max normalisation. This ensures scale consistency
        between binary (0/1) and ordinal (1/2/3) criteria.
        If None, normalisation is computed on filtered_matrix (legacy behaviour).
    """
    # Normalise each criterion to [0, 1] using the global maximum
    # to avoid implicit scale bias between binary and ordinal criteria
    ref_matrix = full_matrix if full_matrix is not None else filtered_matrix
    max_per_criterion = ref_matrix.max(axis=1).replace(0, 1)
    norm_matrix = filtered_matrix.div(max_per_criterion, axis=0)

    scores = {}
    score_breakdown = {}

    for config in norm_matrix.columns:
        partial_scores = {}
        total_score = 0.0

        for criterion in norm_matrix.index:
            if criterion in normalized_weights:
                contribution = (
                    normalized_weights[criterion] *
                    norm_matrix.loc[criterion, config]
                )
                partial_scores[criterion] = contribution
                total_score += contribution

        scores[config] = total_score
        score_breakdown[config] = partial_scores

    results = pd.DataFrame({
        "Configuration": list(scores.keys()),
        "Score": list(scores.values())
    })

    results = results.sort_values("Score", ascending=False).reset_index(drop=True)
    results["Rank"] = results.index + 1
    results = results[["Rank", "Configuration", "Score"]]

    contribution_matrix = pd.DataFrame(score_breakdown)

    return results, score_breakdown, contribution_matrix


def plot_ranking(results: pd.DataFrame, title: str = "Configuration ranking"):
    """
    Plot a horizontal bar chart of the final configuration ranking.
    """

    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.barh(
        results["Configuration"][::-1],
        results["Score"][::-1],
        edgecolor="white",
        linewidth=0.5
    )

    for bar, score in zip(bars, results["Score"][::-1]):
        ax.text(
            bar.get_width() + 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{score:.3f}",
            va="center",
            fontsize=10
        )

    ax.set_xlabel("Score", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlim(0, results["Score"].max() * 1.15)

    plt.tight_layout()
    plt.show()

    return fig


def plot_weights(weights: dict[str, float], title: str = "Normalized weight distribution"):
    """
    Plot a horizontal bar chart of normalized criterion weights.
    """

    fig, ax = plt.subplots(figsize=(10, 6))

    sorted_weights = dict(
        sorted(weights.items(), key=lambda x: x[1], reverse=True)
    )

    bars = ax.barh(
        list(sorted_weights.keys())[::-1],
        list(sorted_weights.values())[::-1],
        edgecolor="white"
    )

    for bar, weight in zip(bars, list(sorted_weights.values())[::-1]):
        ax.text(
            bar.get_width() + 0.005,
            bar.get_y() + bar.get_height() / 2,
            f"{weight:.3f}",
            va="center",
            fontsize=9
        )

    ax.set_xlabel("Normalized weight", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.show()

    return fig


print("Score calculation and plotting functions defined correctly")

Score calculation and plotting functions defined correctly


In [10]:

# =============================================================================
# USER INTERFACE — REDESIGNED QUESTIONNAIRE + OUTPUT
# =============================================================================

# --- Helper: styled section header ---
def section_header(title, subtitle="", color="#2c7fb8"):
    """Return an HTML widget for a styled section header."""
    html = f"""
    <div style="margin: 18px 0 8px 0; padding: 10px 16px; 
                border-left: 4px solid {color}; background: #f7f9fb;">
        <b style="font-size: 14px; color: {color};">{title}</b>
        {'<br><span style="font-size: 12px; color: #666;">' + subtitle + '</span>' if subtitle else ''}
    </div>"""
    return widgets.HTML(html)

def question_label(code, text, hint=""):
    """Return an HTML widget for a question label with optional hint."""
    html = f"""
    <div style="margin: 10px 0 2px 0;">
        <b style="font-size: 13px;">{code}.</b>
        <span style="font-size: 13px;">{text}</span>
        {'<br><span style="font-size: 11px; color: #888; margin-left: 24px;">' + hint + '</span>' if hint else ''}
    </div>"""
    return widgets.HTML(html)


# --- Widget style constants ---
toggle_style = {"description_width": "0px", "button_width": "auto"}
toggle_layout = widgets.Layout(width="auto")
radio_style = {"description_width": "0px"}
radio_layout = widgets.Layout(width="auto")

# =============================================================================
# PHASE 0 — Manufacturing constraints
# =============================================================================
w_integration = widgets.RadioButtons(
    options=[
        ("Fully printable in PLA (no external components)", "fully_printable"),
        ("PLA + simple components (screws, springs, cables, pins)", "simple_components"),
        ("PLA + complex mechanical components (bearings, gears, actuators)", "complex_components")
    ],
    description="",
    style=radio_style,
    layout=widgets.Layout(width="700px")
)

# =============================================================================
# SECTION A — Movement context
# =============================================================================
w_A1 = widgets.RadioButtons(
    options=[
        ("Flexion / extension", "flexion_extension"),
        ("Rotation (e.g. pronation-supination)", "rotation"),
        ("Combination of movements", "combination"),
        ("Elevation or complex spatial motion", "elevation")
    ],
    description="", style=radio_style, layout=widgets.Layout(width="700px")
)

w_A2 = widgets.ToggleButtons(
    options=[("Reduced", "reduced"), ("Medium", "medium"), ("Wide", "wide")],
    description="", style=toggle_style, layout=toggle_layout
)

w_A3 = widgets.ToggleButtons(
    options=[("Uniaxial (1 plane)", "uniaxial"), ("Multiplanar / complex", "multiplanar")],
    description="", style=toggle_style, layout=toggle_layout
)

# =============================================================================
# SECTION B — Orthosis function
# =============================================================================
w_B1 = widgets.RadioButtons(
    options=[
        ("Assist the movement", "assist"),
        ("Resist the movement", "resist"),
        ("Guide the movement along a trajectory", "guide"),
        ("Return the segment to initial position", "return_to_position"),
        ("Maintain a specific trajectory", "maintain_trajectory")
    ],
    description="", style=radio_style, layout=widgets.Layout(width="700px")
)

w_B2 = widgets.ToggleButtons(
    options=[("Yes", "yes"), ("No", "no")],
    description="", style=toggle_style, layout=toggle_layout
)

# =============================================================================
# SECTION C — Technical requirements
# =============================================================================
w_C1 = widgets.RadioButtons(
    options=[
        ("Constant throughout ROM", "constant"),
        ("Variable with joint angle", "variable_with_angle"),
        ("Only in a partial range of the ROM", "partial_range")
    ],
    description="", style=radio_style, layout=widgets.Layout(width="700px")
)

w_C2 = widgets.ToggleButtons(
    options=[("Yes — remote", "yes"), ("No — local", "no")],
    description="", style=toggle_style, layout=toggle_layout
)

w_C3 = widgets.ToggleButtons(
    options=[("Yes — critical", "yes"), ("No — not critical", "no")],
    description="", style=toggle_style, layout=toggle_layout
)

w_C4 = widgets.ToggleButtons(
    options=[("High", "high"), ("Medium", "medium"), ("Low", "low")],
    description="", style=toggle_style, layout=toggle_layout
)

w_C5 = widgets.ToggleButtons(
    options=[("High", "high"), ("Medium", "medium"), ("Low", "low")],
    description="", style=toggle_style, layout=toggle_layout
)

w_C6 = widgets.ToggleButtons(
    options=[("Rotation", "rotation"), ("Translation", "translation"), ("Combined", "combined")],
    description="", style=toggle_style, layout=toggle_layout
)

w_C7 = widgets.ToggleButtons(
    options=[("Low (simple)", "low"), ("Medium", "medium"), ("High (acceptable)", "high")],
    description="", style=toggle_style, layout=toggle_layout
)

w_C8 = widgets.ToggleButtons(
    options=[("Passive (no external power)", "passive"), ("Active (external power)", "active")],
    description="", style=toggle_style, layout=toggle_layout
)

# =============================================================================
# RUN BUTTON
# =============================================================================
run_button = widgets.Button(
    description="  Run DSS",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="200px", height="40px")
)

output = widgets.Output()


# =============================================================================
# DSS EXECUTION FUNCTION (with Problem 4 fix: score-level penalty for 0-R2)
# =============================================================================

def run_dss(button):
    """Execute the full DSS pipeline and display results."""
    with output:
        output.clear_output()

        # --- Collect user inputs ---
        user_inputs = {
            "integration_level": w_integration.value,
            "A1_movement": w_A1.value,
            "A2_rom": w_A2.value,
            "A3_motion_type": w_A3.value,
            "B1_action": w_B1.value,
            "B2_force_required": w_B2.value,
            "C1_assistance_type": w_C1.value,
            "C2_remote_transmission": w_C2.value,
            "C3_low_distal_mass": w_C3.value,
            "C4_adaptability": w_C4.value,
            "C5_precision": w_C5.value,
            "C6_movement_type": w_C6.value,
            "C7_complexity": w_C7.value,
            "Actuation_type": w_C8.value
        }

        # =====================================================================
        # PHASE 0: Manufacturing filter
        # =====================================================================
        display(HTML("""
        <div style="margin: 12px 0; padding: 12px 16px; border-left: 4px solid #2c7fb8; 
                    background: #eef4fa;">
            <b style="font-size: 14px; color: #2c7fb8;">PHASE 0 — Manufacturing filter</b>
        </div>"""))

        eligible_configs, eliminated_configs, complexity_adjusted_configs = \
            apply_manufacturing_filter(PLA_filter, user_inputs["integration_level"])

        filtered_matrix = matrix[eligible_configs]
        # NOTE: No longer calling preprocess_matrix_for_manufacturing.
        # The 0-R2 penalty is applied at score level after Phase 2.

        # Architecture branching
        filtered_matrix, branch_excluded = apply_architecture_branching(
            filtered_matrix, user_inputs
        )

        if filtered_matrix.shape[1] == 0:
            display(HTML("""
            <div style="padding: 12px; background: #fff3cd; border: 1px solid #ffc107; 
                        border-radius: 4px; margin: 8px 0;">
                <b>No configurations remain</b> after applying manufacturing and architectural 
                filters. Consider relaxing one or more constraints.
            </div>"""))
            return

        # Summary table for Phase 0
        level_labels = {
            "fully_printable": "Fully printable in PLA",
            "simple_components": "PLA + simple components",
            "complex_components": "PLA + complex components"
        }

        summary_rows = [
            f"<tr><td style='padding:4px 12px;'><b>Integration level</b></td>"
            f"<td style='padding:4px 12px;'>{level_labels.get(user_inputs['integration_level'], user_inputs['integration_level'])}</td></tr>",
            f"<tr><td style='padding:4px 12px;'><b>Eligible after filter</b></td>"
            f"<td style='padding:4px 12px;'>{len(eligible_configs)} configurations</td></tr>",
        ]
        if eliminated_configs:
            summary_rows.append(
                f"<tr><td style='padding:4px 12px;'><b>Eliminated (manufacturing)</b></td>"
                f"<td style='padding:4px 12px; color:#c0392b;'>{', '.join(eliminated_configs)}</td></tr>"
            )
        if branch_excluded:
            summary_rows.append(
                f"<tr><td style='padding:4px 12px;'><b>Excluded (architectural branching)</b></td>"
                f"<td style='padding:4px 12px; color:#c0392b;'>{', '.join(branch_excluded)}</td></tr>"
            )
        if complexity_adjusted_configs:
            summary_rows.append(
                f"<tr><td style='padding:4px 12px;'><b>Complexity penalty (0-R2, x0.80)</b></td>"
                f"<td style='padding:4px 12px; color:#e67e22;'>{', '.join(complexity_adjusted_configs)}</td></tr>"
            )
        summary_rows.append(
            f"<tr style='background:#eef4fa;'><td style='padding:4px 12px;'><b>Final candidates</b></td>"
            f"<td style='padding:4px 12px;'><b>{', '.join(filtered_matrix.columns)}</b></td></tr>"
        )

        display(HTML(
            "<table style='border-collapse:collapse; margin: 8px 0; font-size:13px;'>"
            + "".join(summary_rows) + "</table>"
        ))

        # =====================================================================
        # PHASE 1: Rule-based weight adjustment
        # =====================================================================
        display(HTML("""
        <div style="margin: 16px 0 8px 0; padding: 12px 16px; border-left: 4px solid #27ae60; 
                    background: #eafaf1;">
            <b style="font-size: 14px; color: #27ae60;">PHASE 1 — Rule-based weight adjustment</b>
        </div>"""))

        normalized_weights, activated_rules, detail = rule_engine.apply_rules(user_inputs)

        # Activated rules as compact list
        rules_html = "".join(
            f"<li style='margin:2px 0; font-size:12px;'>"
            f"<code>{r.id}</code> — {r.description}</li>"
            for r in activated_rules
        )
        display(HTML(
            f"<p style='font-size:13px;'><b>{len(activated_rules)} rules activated:</b></p>"
            f"<ul style='margin:4px 0 12px 20px;'>{rules_html}</ul>"
        ))

        # Weight detail table (styled)
        display(
            detail.style.format({
                "Base_weight": "{:.2f}",
                "Accumulated_multiplier": "{:.3f}",
                "Raw_weight": "{:.3f}",
                "Capped_weight": "{:.3f}",
                "Normalized_weight": "{:.4f}"
            }).background_gradient(subset=["Normalized_weight"], cmap="YlGn")
            .set_caption("Weight calculation trace")
        )

        plot_weights(normalized_weights, "Normalized criterion weights")

        # =====================================================================
        # PHASE 2: Score calculation + 0-R2 penalty
        # =====================================================================
        display(HTML("""
        <div style="margin: 16px 0 8px 0; padding: 12px 16px; border-left: 4px solid #8e44ad; 
                    background: #f5eef8;">
            <b style="font-size: 14px; color: #8e44ad;">PHASE 2 — Score calculation and ranking</b>
        </div>"""))

        results, score_breakdown, contribution_matrix = calculate_scores(
            filtered_matrix, normalized_weights, matrix
        )

        # Apply 0-R2 complexity penalty at score level (Problem 4 fix)
        if complexity_adjusted_configs:
            raw_scores = dict(zip(results["Configuration"], results["Score"]))
            adjusted_scores = apply_manufacturing_complexity_penalty(
                raw_scores, score_breakdown, complexity_adjusted_configs, penalty_factor=0.80
            )
            # Update results with adjusted scores
            results["Score"] = results["Configuration"].map(adjusted_scores)
            results = results.sort_values("Score", ascending=False).reset_index(drop=True)
            results["Rank"] = results.index + 1
            results = results[["Rank", "Configuration", "Score"]]

        display(
            results.style.format({"Score": "{:.4f}"})
            .background_gradient(subset=["Score"], cmap="YlGn")
            .set_caption("Configuration ranking")
        )

        plot_ranking(results, "Mechanical configuration ranking")

        # =====================================================================
        # RECOMMENDATIONS
        # =====================================================================
        display(HTML("""
        <div style="margin: 16px 0 8px 0; padding: 12px 16px; border-left: 4px solid #e67e22; 
                    background: #fef9e7;">
            <b style="font-size: 14px; color: #e67e22;">TOP RECOMMENDATIONS</b>
        </div>"""))

        top_3 = results.head(3)
        rec_html = ""
        for _, row in top_3.iterrows():
            rank = int(row["Rank"])
            config = row["Configuration"]
            score = row["Score"]
            medal = ["🥇", "🥈", "🥉"][rank - 1] if rank <= 3 else f"{rank}."

            rec_html += f"""
            <div style="display:flex; align-items:center; padding: 8px 12px; margin: 4px 0; 
                        background: {'#eafaf1' if rank == 1 else '#f8f9fa'}; 
                        border-radius: 6px; border: 1px solid {'#27ae60' if rank == 1 else '#dee2e6'};">
                <span style="font-size: 22px; margin-right: 12px;">{medal}</span>
                <div>
                    <b style="font-size: 14px;">{config}</b>
                    <span style="font-size: 12px; color: #666; margin-left: 12px;">
                        Score: {score:.4f}
                    </span>
                </div>
            </div>"""

        display(HTML(rec_html))

        # Caveat for A6 if it appears in top results
        a6_names = ["A6_Flexible_shell", "A6_Flex_shell", "Flexible_shell", "A6"]
        top_configs = list(top_3["Configuration"])
        if any(a6 in cfg for cfg in top_configs for a6 in a6_names):
            display(HTML("""
            <div style="padding: 10px 14px; margin: 10px 0; background: #fff3cd; 
                        border: 1px solid #ffc107; border-radius: 4px; font-size: 12px;">
                <b>⚠ PLA fatigue caveat for A6 (Flexible shell / Leaf-spring):</b> 
                PLA shows limited fatigue life under cyclic bending loads exceeding 50% 
                of tensile strength. For ankle-level loads with thousands of gait cycles, 
                standard PLA is insufficient — validate with material testing or consider 
                PETG/PA alternatives. This archetype is viable in PLA for low-load 
                applications (hand, wrist, finger) and conceptual prototyping.
            </div>"""))

        # Complexity penalty note
        if complexity_adjusted_configs:
            penalized_in_top = [c for c in top_configs if c in complexity_adjusted_configs]
            if penalized_in_top:
                display(HTML(f"""
                <div style="padding: 10px 14px; margin: 10px 0; background: #fef9e7; 
                            border: 1px solid #f39c12; border-radius: 4px; font-size: 12px;">
                    <b>ℹ Manufacturing note (Rule 0-R2):</b> 
                    {', '.join(penalized_in_top)} received a x0.80 penalty on the Complexity_level 
                    contribution because they require complex components under the selected 
                    manufacturing level (PLA + simple components).
                </div>"""))

        display(HTML("""
        <div style="margin-top: 16px; padding: 8px 12px; background: #f8f9fa; 
                    border-radius: 4px; font-size: 11px; color: #888;">
            These outputs are conceptual design suggestions for the selection of transmission 
            mechanism archetypes — not final design solutions. Further engineering analysis 
            is required before clinical implementation.
        </div>"""))


run_button.on_click(run_dss)


# =============================================================================
# DISPLAY QUESTIONNAIRE — Redesigned with labels, hints, and sections
# =============================================================================

display(HTML("""
<div style="padding: 16px 20px; margin: 0 0 16px 0; background: linear-gradient(135deg, #2c7fb8 0%, #1a5276 100%); 
            color: white; border-radius: 8px;">
    <h2 style="margin: 0 0 4px 0; color: white;">DSS — Mechanical Configuration Selection</h2>
    <p style="margin: 0; font-size: 13px; opacity: 0.9;">
        Decision Support System for dynamic orthosis transmission mechanism selection.
        Answer each question and press <b>Run DSS</b>.
    </p>
</div>
"""))

# Phase 0
display(section_header("PHASE 0 — Manufacturing constraints",
                       "Define what fabrication resources are available."))
display(question_label("P0", "What manufacturing integration level is available?",
                       "This determines which archetypes can be considered."))
display(w_integration)

# Section A
display(section_header("SECTION A — Movement context",
                       "Describe the biomechanical movement the orthosis must accommodate.",
                       color="#27ae60"))

display(question_label("A1", "What movement do you want to assist or enable?",
                       "Describe in terms of osteokinematics (bone-level motion)."))
display(w_A1)

display(question_label("A2", "What is the required range of motion (ROM)?",
                       "Reduced: small arc | Medium: typical clinical ROM | Wide: full joint excursion."))
display(w_A2)

display(question_label("A3", "Is the motion uniaxial or multiplanar?",
                       "Uniaxial = 1 plane, 1 DOF | Multiplanar = multiple planes or coupled DOFs."))
display(w_A3)

# Section B
display(section_header("SECTION B — Orthosis function",
                       "Define the mechanical role of the orthosis.",
                       color="#e67e22"))

display(question_label("B1", "What is the intended action of the orthosis?",
                       "This defines the primary mechanical behavior the device must perform."))
display(w_B1)

display(question_label("B2", "Is force or torque generation required?",
                       "Yes = device must generate/transmit force | No = device mainly guides or constrains."))
display(w_B2)

# Section C
display(section_header("SECTION C — Technical requirements",
                       "Specify the engineering constraints and performance expectations.",
                       color="#8e44ad"))

display(question_label("C1", "What kind of assistance profile is needed?",
                       "Constant = uniform support | Variable = angle-dependent | Partial = only in part of ROM."))
display(w_C1)

display(question_label("C2", "Should the mechanism be based on remote transmission?",
                       "Yes = remote transmission is required/preferred | No = no, a local or non-remote solution is preferred."))
display(w_C2)

display(question_label("C3", "Is it important to reduce mass in the distal region?",
                       "Critical for dynamic tasks (gait, repetitive upper-limb) where distal inertia matters."))
display(w_C3)

display(question_label("C4", "What level of adaptability to misalignment is required?",
                       "High = complex joints, axis migration | Low = well-defined, fixed-axis joints."))
display(w_C4)

display(question_label("C5", "What level of movement precision is required?",
                       "High = strict trajectory control | Low = functional assistance is sufficient."))
display(w_C5)

display(question_label("C6", "What type of mechanical movement is expected?",
                       "Rotation = angular motion | Translation = linear | Combined = both."))
display(w_C6)

display(question_label("C7", "What level of design complexity is acceptable?",
                       "Low = simple, robust, easy to fabricate | High = advanced mechanisms acceptable."))
display(w_C7)

display(question_label("C8", "What type of actuation is intended?",
                       "Passive = energy storage/redirection only | Active = motors, actuators, external power."))
display(w_C8)

# Run button
display(HTML("<div style='margin: 20px 0 8px 0;'>"))
display(run_button)
display(HTML("</div>"))
display(output)


HTML(value='\n    <div style="margin: 18px 0 8px 0; padding: 10px 16px; \n                border-left: 4px sol…

HTML(value='\n    <div style="margin: 10px 0 2px 0;">\n        <b style="font-size: 13px;">P0.</b>\n        <s…

RadioButtons(layout=Layout(width='700px'), options=(('Fully printable in PLA (no external components)', 'fully…

HTML(value='\n    <div style="margin: 18px 0 8px 0; padding: 10px 16px; \n                border-left: 4px sol…

HTML(value='\n    <div style="margin: 10px 0 2px 0;">\n        <b style="font-size: 13px;">A1.</b>\n        <s…

...

ToggleButtons(layout=Layout(width='auto'), options=(('Low (simple)', 'low'), ('Medium', 'medium'), ('High (acc…

HTML(value='\n    <div style="margin: 10px 0 2px 0;">\n        <b style="font-size: 13px;">C8.</b>\n        <s…

ToggleButtons(layout=Layout(width='auto'), options=(('Passive (no external power)', 'passive'), ('Active (exte…

Button(button_style='success', description='  Run DSS', icon='play', layout=Layout(height='40px', width='200px…

Output()

In [11]:
# =============================================================================
# TEST MODE — predefined clinical cases
# Run this cell independently to test without filling the questionnaire.
# =============================================================================

def run_test_pipeline(user_inputs: dict, case_label: str = "") -> None:
    """
    Run the full DSS pipeline for a given user_inputs dict and print
    a compact ranked output. All dependencies (matrix, rule_engine, etc.)
    must be defined before this cell is executed.
    """
    medals = ["1.", "2.", "3.", "4.", "5."]

    if case_label:
        print("=" * 65)
        print(f"  {case_label}")
        print("=" * 65)

    # Phase 0 — manufacturing filter
    eligible_configs, eliminated_configs, complexity_adjusted_configs = \
        apply_manufacturing_filter(PLA_filter, user_inputs["integration_level"])

    if eliminated_configs:
        print(f"  [Phase 0] Eliminated: {eliminated_configs}")
    if complexity_adjusted_configs:
        print(f"  [Phase 0] Complexity penalty (x0.80): {complexity_adjusted_configs}")

    filtered_matrix = matrix[eligible_configs]

    # Architecture branching
    filtered_matrix, branch_excluded = apply_architecture_branching(
        filtered_matrix, user_inputs
    )
    if branch_excluded:
        print(f"  [Branching] Excluded: {branch_excluded}")

    if filtered_matrix.shape[1] == 0:
        print("  No candidates remain after filters.")
        return

    # Phase 1 — rules
    normalized_weights, activated_rules, _ = rule_engine.apply_rules(user_inputs)
    print(f"  [Rules] {len(activated_rules)} activated: "
          f"{', '.join(r.id for r in activated_rules)}")

    # Phase 2 — scores
    results, score_breakdown, _ = calculate_scores(
        filtered_matrix, normalized_weights, matrix
    )

    # 0-R2 complexity penalty at score level
    if complexity_adjusted_configs:
        raw_scores = dict(zip(results["Configuration"], results["Score"]))
        adjusted = apply_manufacturing_complexity_penalty(
            raw_scores, score_breakdown, complexity_adjusted_configs
        )
        results["Score"] = results["Configuration"].map(adjusted)
        results = results.sort_values("Score", ascending=False).reset_index(drop=True)
        results["Rank"] = results.index + 1

    # Print ranking
    print()
    for _, row in results.head(5).iterrows():
        rank = int(row["Rank"])
        m = medals[rank - 1] if rank <= len(medals) else f"{rank}."
        print(f"    {m:<4} {row['Configuration']:<52}  {row['Score']:.4f}")
    print()


TEST_CASES = {
    # ── Original validation cases ─────────────────────────────────────────────
    "Case 1 — Shoulder elevation assist (passive, PLA+simple)": {
        "integration_level": "simple_components",
        "A1_movement":        "elevation",
        "A2_rom":             "wide",
        "A3_motion_type":     "multiplanar",
        "B1_action":          "assist",
        "B2_force_required":  "no",
        "C1_assistance_type": "variable_with_angle",
        "C2_remote_transmission": "yes",
        "C3_low_distal_mass": "yes",
        "C4_adaptability":    "high",
        "C5_precision":       "medium",
        "C6_movement_type":   "combined",
        "C7_complexity":      "medium",
        "Actuation_type":     "passive",
    },
    "Case 2 — Elbow flexion assist (passive, PLA, no mass constraint)": {
        "integration_level": "simple_components",
        "A1_movement":        "flexion_extension",
        "A2_rom":             "medium",
        "A3_motion_type":     "uniaxial",
        "B1_action":          "assist",
        "B2_force_required":  "no",
        "C1_assistance_type": "constant",
        "C2_remote_transmission": "no",
        "C3_low_distal_mass": "no",
        "C4_adaptability":    "low",
        "C5_precision":       "medium",
        "C6_movement_type":   "rotation",
        "C7_complexity":      "low",
        "Actuation_type":     "passive",
    },
    "Case 3 — Hip flexion during gait (passive, minimise thigh mass)": {
        "integration_level": "simple_components",
        "A1_movement":        "combination",
        "A2_rom":             "medium",
        "A3_motion_type":     "multiplanar",
        "B1_action":          "assist",
        "B2_force_required":  "no",
        "C1_assistance_type": "partial_range",
        "C2_remote_transmission": "yes",
        "C3_low_distal_mass": "yes",
        "C4_adaptability":    "medium",
        "C5_precision":       "low",
        "C6_movement_type":   "rotation",
        "C7_complexity":      "medium",
        "Actuation_type":     "passive",
    },
    "Case 4 — Drop foot AFO (passive, lightweight, PLA)": {
        "integration_level": "fully_printable",
        "A1_movement":        "flexion_extension",
        "A2_rom":             "reduced",
        "A3_motion_type":     "uniaxial",
        "B1_action":          "return_to_position",
        "B2_force_required":  "no",
        "C1_assistance_type": "partial_range",
        "C2_remote_transmission": "no",
        "C3_low_distal_mass": "yes",
        "C4_adaptability":    "medium",
        "C5_precision":       "medium",
        "C6_movement_type":   "rotation",
        "C7_complexity":      "low",
        "Actuation_type":     "passive",
    },
    "Case 5 — Wrist extension assist (lightweight, PLA+simple)": {
        "integration_level": "simple_components",
        "A1_movement":        "combination",
        "A2_rom":             "medium",
        "A3_motion_type":     "multiplanar",
        "B1_action":          "guide",
        "B2_force_required":  "no",
        "C1_assistance_type": "partial_range",
        "C2_remote_transmission": "no",
        "C3_low_distal_mass": "yes",
        "C4_adaptability":    "medium",
        "C5_precision":       "medium",
        "C6_movement_type":   "combined",
        "C7_complexity":      "low",
        "Actuation_type":     "passive",
    },
    "Case 6 — Elbow flexion active (motor, compact, low complexity)": {
        "integration_level": "complex_components",
        "A1_movement":        "flexion_extension",
        "A2_rom":             "medium",
        "A3_motion_type":     "uniaxial",
        "B1_action":          "assist",
        "B2_force_required":  "yes",
        "C1_assistance_type": "constant",
        "C2_remote_transmission": "no",
        "C3_low_distal_mass": "no",
        "C4_adaptability":    "low",
        "C5_precision":       "high",
        "C6_movement_type":   "rotation",
        "C7_complexity":      "high",
        "Actuation_type":     "active",
    },
    # ── Control cases ─────────────────────────────────────────────────────────
    "Control 1 — Cable/tendon must win (hand extension, active, remote)": {
        "integration_level": "complex_components",
        "A1_movement":        "combination",
        "A2_rom":             "medium",
        "A3_motion_type":     "multiplanar",
        "B1_action":          "assist",
        "B2_force_required":  "yes",
        "C1_assistance_type": "variable_with_angle",
        "C2_remote_transmission": "yes",
        "C3_low_distal_mass": "yes",
        "C4_adaptability":    "high",
        "C5_precision":       "medium",
        "C6_movement_type":   "combined",
        "C7_complexity":      "medium",
        "Actuation_type":     "active",
    },
    "Control 2 — Local rigid rotary must win (active coaxial elbow)": {
        "integration_level": "complex_components",
        "A1_movement":        "flexion_extension",
        "A2_rom":             "wide",
        "A3_motion_type":     "uniaxial",
        "B1_action":          "assist",
        "B2_force_required":  "yes",
        "C1_assistance_type": "constant",
        "C2_remote_transmission": "no",
        "C3_low_distal_mass": "no",
        "C4_adaptability":    "low",
        "C5_precision":       "high",
        "C6_movement_type":   "rotation",
        "C7_complexity":      "high",
        "Actuation_type":     "active",
    },
    "Control 3 — Bowden cable must win (active shoulder, motor on trunk)": {
        "integration_level": "complex_components",
        "A1_movement":        "elevation",
        "A2_rom":             "wide",
        "A3_motion_type":     "multiplanar",
        "B1_action":          "assist",
        "B2_force_required":  "yes",
        "C1_assistance_type": "variable_with_angle",
        "C2_remote_transmission": "yes",
        "C3_low_distal_mass": "yes",
        "C4_adaptability":    "high",
        "C5_precision":       "low",
        "C6_movement_type":   "combined",
        "C7_complexity":      "medium",
        "Actuation_type":     "active",
    },
    "Control 4 — Four-bar must win (knee trajectory guidance, passive)": {
        "integration_level": "simple_components",
        "A1_movement":        "flexion_extension",
        "A2_rom":             "wide",
        "A3_motion_type":     "uniaxial",
        "B1_action":          "guide",
        "B2_force_required":  "no",
        "C1_assistance_type": "constant",
        "C2_remote_transmission": "no",
        "C3_low_distal_mass": "no",
        "C4_adaptability":    "medium",
        "C5_precision":       "high",
        "C6_movement_type":   "rotation",
        "C7_complexity":      "medium",
        "Actuation_type":     "passive",
    },
    "Control 5 — Flexible shell must win (AFO spasticity, PLA, simple)": {
        "integration_level": "fully_printable",
        "A1_movement":        "flexion_extension",
        "A2_rom":             "reduced",
        "A3_motion_type":     "uniaxial",
        "B1_action":          "return_to_position",
        "B2_force_required":  "no",
        "C1_assistance_type": "partial_range",
        "C2_remote_transmission": "no",
        "C3_low_distal_mass": "yes",
        "C4_adaptability":    "medium",
        "C5_precision":       "low",
        "C6_movement_type":   "rotation",
        "C7_complexity":      "low",
        "Actuation_type":     "passive",
    },
    "Control 6 — Cam-spring must win (passive shoulder, variable profile)": {
        "integration_level": "complex_components",
        "A1_movement":        "elevation",
        "A2_rom":             "wide",
        "A3_motion_type":     "multiplanar",
        "B1_action":          "assist",
        "B2_force_required":  "no",
        "C1_assistance_type": "variable_with_angle",
        "C2_remote_transmission": "yes",
        "C3_low_distal_mass": "yes",
        "C4_adaptability":    "medium",
        "C5_precision":       "low",
        "C6_movement_type":   "combined",
        "C7_complexity":      "medium",
        "Actuation_type":     "passive",
    },
}

# ── Widget map: test key → (widget, value_key) ────────────────────────────────
WIDGET_MAP = {
    "integration_level":      w_integration,
    "A1_movement":            w_A1,
    "A2_rom":                 w_A2,
    "A3_motion_type":         w_A3,
    "B1_action":              w_B1,
    "B2_force_required":      w_B2,
    "C1_assistance_type":     w_C1,
    "C2_remote_transmission": w_C2,
    "C3_low_distal_mass":     w_C3,
    "C4_adaptability":        w_C4,
    "C5_precision":           w_C5,
    "C6_movement_type":       w_C6,
    "C7_complexity":          w_C7,
    "Actuation_type":         w_C8,
}

# ── UI ─────────────────────────────────────────────────────────────────────────
test_selector = widgets.Dropdown(
    options=list(TEST_CASES.keys()),
    description="",
    layout=widgets.Layout(width="680px"),
    style={"description_width": "0px"},
)

test_output = widgets.Output()

run_test_button = widgets.Button(
    description="  Run test case",
    button_style="info",
    icon="flask",
    layout=widgets.Layout(width="200px", height="40px"),
)

run_all_button = widgets.Button(
    description="  Run ALL cases",
    button_style="warning",
    icon="list",
    layout=widgets.Layout(width="200px", height="40px"),
)


# Two output modes per button:
#   "Full DSS output" — updates widgets + fires run_dss (rich HTML output)
#   "Compact ranking"  — runs run_test_pipeline directly (plain text, fast)

mode_toggle = widgets.ToggleButtons(
    options=[("Full DSS output", "full"), ("Compact ranking", "compact")],
    description="",
    style={"description_width": "0px", "button_width": "160px"},
    layout=widgets.Layout(margin="6px 0"),
)


def _run_full(case_name, case_inputs):
    """Update widgets and fire run_dss for rich HTML output."""
    for key, widget in WIDGET_MAP.items():
        if key in case_inputs:
            widget.value = case_inputs[key]

    inputs_html = " | ".join(
        f"<code>{k}: {v}</code>" for k, v in case_inputs.items()
    )
    display(HTML(f"""
    <div style="padding: 10px 16px; margin-bottom: 10px;
                background: linear-gradient(135deg, #1a5276 0%, #2e74b5 100%);
                border-radius: 6px; color: white;">
        <b style="font-size: 14px;">TEST — {case_name}</b>
    </div>
    <div style="font-size: 11px; color: #666; margin-bottom: 8px;">
        {inputs_html}
    </div>"""))
    run_dss(None)


def _run_compact(case_name, case_inputs):
    """Run the pipeline directly and print a compact ranked table."""
    run_test_pipeline(case_inputs, case_label=case_name)


def on_run_test(button):
    with test_output:
        test_output.clear_output()
        case_name = test_selector.value
        case_inputs = TEST_CASES[case_name]
        if mode_toggle.value == "full":
            _run_full(case_name, case_inputs)
        else:
            _run_compact(case_name, case_inputs)


def on_run_all(button):
    with test_output:
        test_output.clear_output()

        if mode_toggle.value == "full":
            display(HTML("""
            <div style="padding: 10px 16px; margin-bottom: 14px;
                        background: linear-gradient(135deg, #7d6608 0%, #d4ac0d 100%);
                        border-radius: 6px; color: white;">
                <b style="font-size: 14px;">ALL CASES — Full DSS output</b>
            </div>"""))
            for case_name, case_inputs in TEST_CASES.items():
                display(HTML(f"""
                <div style="margin: 18px 0 4px 0; padding: 8px 14px;
                            border-left: 5px solid #2e74b5; background: #eef4fa;">
                    <b style="font-size: 13px; color: #1a5276;">▶ {case_name}</b>
                </div>"""))
                _run_full(case_name, case_inputs)
                display(HTML(
                    "<hr style='border:none; border-top:1px solid #ddd; margin:14px 0;'>"
                ))
        else:
            # Compact mode: all cases in one clean text block
            display(HTML("""
            <div style="padding: 10px 16px; margin-bottom: 14px;
                        background: linear-gradient(135deg, #117a65 0%, #1abc9c 100%);
                        border-radius: 6px; color: white;">
                <b style="font-size: 14px;">ALL CASES — Compact ranking</b><br>
                <span style="font-size: 12px; opacity:0.9;">
                    Top 5 candidates per case
                </span>
            </div>"""))
            for case_name, case_inputs in TEST_CASES.items():
                run_test_pipeline(case_inputs, case_label=case_name)


run_test_button.on_click(on_run_test)
run_all_button.on_click(on_run_all)

# ── Display test panel ────────────────────────────────────────────────────────
display(HTML("""
<div style="padding: 14px 18px; margin: 0 0 14px 0;
            background: linear-gradient(135deg, #1a5276 0%, #117a65 100%);
            color: white; border-radius: 8px;">
    <h3 style="margin: 0 0 4px 0; color: white;">TEST MODE</h3>
    <p style="margin: 0; font-size: 12px; opacity: 0.9;">
        Select a predefined case and click <b>Run test case</b>, or run all cases at once.
        Widget values in the questionnaire above will be updated automatically.
    </p>
</div>
"""))

display(widgets.HTML("<b style='font-size:13px;'>Output mode:</b>"))
display(mode_toggle)
display(widgets.HTML("<b style='font-size:13px;'>Select test case:</b>"))
display(test_selector)
display(widgets.HBox([run_test_button, run_all_button],
                     layout=widgets.Layout(gap="12px", margin="10px 0")))
display(test_output)


HTML(value="<b style='font-size:13px;'>Output mode:</b>")

ToggleButtons(layout=Layout(margin='6px 0'), options=(('Full DSS output', 'full'), ('Compact ranking', 'compac…

HTML(value="<b style='font-size:13px;'>Select test case:</b>")

Dropdown(layout=Layout(width='680px'), options=('Case 1 — Shoulder elevation assist (passive, PLA+simple)', 'C…

Output()